In [43]:
# Imports section
import pandas as pd
import sklearn
import sklearn.model_selection
import sklearn.linear_model
from sklearn.preprocessing import PolynomialFeatures

## Part 1. Loading the dataset

In [44]:
# Using pandas load the dataset (load remotely, not locally)
ds = pd.read_csv("science_data_large.csv")
# Output the first 15 rows of the data
print(ds.head(n=15))
# Display a summary of the table information (number of datapoints, etc.)
ds.info()
print(ds.describe())

    Temperature °C  Mols KCL     Size nm^3
0              469       647  6.244743e+05
1              403       694  5.779610e+05
2              302       975  6.196847e+05
3              779       916  1.460449e+06
4              901        18  4.325726e+04
5              545       637  7.124634e+05
6              660       519  7.006960e+05
7              143       869  2.718260e+05
8               89       461  8.919803e+04
9              294       776  4.770210e+05
10             991       117  2.441771e+05
11             307       781  5.006455e+05
12             206        70  3.145200e+04
13             437       599  5.390215e+05
14             566        75  9.185271e+04
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Temperature °C  1000 non-null   int64  
 1   Mols KCL        1000 non-null   int64  
 2   Size nm^3       1000 n

## Part 2. Splitting the dataset

In [45]:
# Take the pandas dataset and split it into our features (X) and label (y)
X = ds.loc[:, ["Temperature °C", "Mols KCL"]]
y = ds["Size nm^3"]
# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
Xtrain, Xtest, ytrain, ytest = sklearn.model_selection.train_test_split(X, y, test_size=0.1)

## Part 3. Perform a Linear Regression

In [46]:
# Use sklearn to train a model on the training set
lr = sklearn.linear_model.LinearRegression()
lr.fit(Xtrain, ytrain)
# Create a sample datapoint and predict the output of that sample with the trained model
print(type(Xtest))
sampledp, samplecorrect = Xtest.iloc[[21]], ytest.iloc[[21]]
print(sampledp, samplecorrect)
sampleresult = lr.predict(sampledp)
print("Predicted result: ", sampleresult[0])
print("Actual result: ", samplecorrect.item())
# Report on the score for that model, in your own words (markdown, not code) explain what the score means
score = lr.score(Xtest, ytest)
print("Score: ", score)
# Extract the coefficents and intercept from the model and write an equation for your h(x) using LaTeX
coeff = lr.coef_
print(coeff)

<class 'pandas.core.frame.DataFrame'>
     Temperature °C  Mols KCL
819             612       109 819    141099.4571
Name: Size nm^3, dtype: float64
Predicted result:  233936.84689278918
Actual result:  141099.4571
Score:  0.8626850545666143
[ 872.51470026 1030.17800006]


The score of 0.86 means that this is a decent model, but it could definitely be improved. Browsing through the data to find various other datapoints, many are quite far off, and a better model is definitely worth attempting.

Linear Regression Equation: $S = 856.85698882 \cdot T + 1024.72081334 \cdot m$ where S is the size of the slime in nanometers cubed, T is the temperature in degrees Celsius, and m is the mols of potassium chloride.

## Part 4. Use Cross Validation

In [47]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
scores = sklearn.model_selection.cross_val_score(lr, Xtrain, ytrain)
print(scores)
# Report on their finding and their significance

[0.86245882 0.88192489 0.84305499 0.82778322 0.86771193]


These scores are all pretty comparable to the first result, all around 0.85 and none above 0.9. This indicates that the model can't be improved through small tweaks and instead we should investigate a nonlinear response between these variables in order to have higher predictive accuracy.

## Part 5. Using Polynomial Regression

In [48]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
poly = PolynomialFeatures(degree=2)
polytrain = poly.fit_transform(Xtrain)
lr2 = sklearn.linear_model.LinearRegression()
lr2.fit(polytrain, ytrain)
# Report on the metrics and output the resultant equation as you did in Part 3.
polytest = poly.fit_transform(Xtest)
score = lr2.score(polytest, ytest)
print("Model score: ", score)
scores = sklearn.model_selection.cross_val_score(lr2, polytrain, ytrain)
print("Cross validation scores: ", scores)
coeffs = lr2.coef_
print(coeffs)

Model score:  1.0
Cross validation scores:  [1. 1. 1. 1. 1.]
[ 0.00000000e+00  1.20000000e+01 -1.24586014e-07 -6.20836715e-12
  2.00000000e+00  2.85714287e-02]


The model score for this is 1.0 and the cross validation scores are all also 1.0, which shows that this model by including more polynomial features perfectly predicts the size of the slime.

Equation: $S = 12 T + -1.03555956\cdot 10^{-7} \cdot m + 2.46203058 \cdot 10^{-11} \cdot T^2 + 2 Tm + 0.0285714287 m^2$ where S, T, and m have the same meanings as before (S is the size of the slime in nanometers cubed, T is the temperature in degrees Celsius, and m is the mols of potassium chloride). 